# 02 — Embedding Analysis

Inspect embeddings, PCA variance, and KNN densities produced by Part 2.

**Sections**
1. Configuration & pipeline run
2. PCA variance curve
3. Embedding space (2-D UMAP projection)
4. Density distributions per group and K
5. Density ratio (Russian vs all) analysis
6. Cross-group density heatmap

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.embedding.pipeline import run
from src.embedding.pca import load_pca, plot_variance_curve

## 1. Configuration & Pipeline Run

Set `RUN_PIPELINE = True` the first time (generates all outputs).  
Set it to `False` on subsequent runs to skip straight to analysis.

In [ ]:
CONFIG_PATH = "../configs/datasets.yaml"
RUN_PIPELINE = False  # set True to regenerate embeddings/densities

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

emb_cfg = cfg["embedding"]
MODEL_NAME    = emb_cfg["models"][0]
K_VALUES      = emb_cfg["k_values"]
EMBEDDINGS_ROOT = emb_cfg["output_root"]
PREPROCESSED_ROOT = cfg["preprocessing"]["output_root"]
DATASETS      = list(cfg["datasets"].keys())

MODEL_SLUG = MODEL_NAME.replace("/", "_").replace("\\", "_")

print("Model     :", MODEL_NAME)
print("K values  :", K_VALUES)
print("Datasets  :", DATASETS)

In [ ]:
if RUN_PIPELINE:
    run(config_path=CONFIG_PATH)
else:
    print("Skipping pipeline run — loading existing outputs.")

## 2. PCA Variance Curve

In [ ]:
pca_path = os.path.join(EMBEDDINGS_ROOT, "_pca", MODEL_SLUG, "shared.pkl")
pca = load_pca(pca_path)

evr = pca.explained_variance_ratio_
cumvar = np.cumsum(evr)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(cumvar) + 1), cumvar, linewidth=1.5)
ax.axhline(0.95, color="grey", linestyle=":", label="95% threshold")
ax.axvline(pca.n_components_, color="tab:red", linestyle="--",
           label=f"Selected: {pca.n_components_} components")
ax.set_xlabel("Number of PCA components")
ax.set_ylabel("Cumulative explained variance")
ax.set_title("PCA Explained Variance — shared ToxiGen PCA")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Components used : {pca.n_components_}")
print(f"Variance captured: {cumvar[pca.n_components_ - 1]:.2%}")

## 3. Embedding Space — 2-D UMAP Projection

Requires `umap-learn` (`pip install umap-learn`).  
We subsample to 5 000 points per dataset to keep it fast.

In [ ]:
try:
    import umap
    from src.embedding.pca import project

    UMAP_SAMPLE = 5_000
    frames = []
    for ds in DATASETS:
        csv = pd.read_csv(os.path.join(PREPROCESSED_ROOT, ds, "full.csv"))
        emb = np.load(os.path.join(EMBEDDINGS_ROOT, ds, MODEL_SLUG, "full.npy"))
        emb_r = project(pca, emb)
        sample_idx = np.random.choice(len(csv), min(UMAP_SAMPLE, len(csv)), replace=False)
        sub_df = csv.iloc[sample_idx].copy()
        sub_df["_emb"] = list(emb_r[sample_idx])
        frames.append(sub_df)

    df_all = pd.concat(frames, ignore_index=True)
    X = np.stack(df_all["_emb"].values)

    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=30)
    coords = reducer.fit_transform(X)

    df_all["umap_x"] = coords[:, 0]
    df_all["umap_y"] = coords[:, 1]

    fig, ax = plt.subplots(figsize=(12, 8))
    for grp, sub in df_all.groupby("group"):
        ax.scatter(sub["umap_x"], sub["umap_y"], s=2, alpha=0.4, label=grp)
    ax.legend(markerscale=4, fontsize=7, ncol=3)
    ax.set_title("UMAP projection — all groups")
    plt.tight_layout()
    plt.show()

except ImportError:
    print("umap-learn not installed. Run: pip install umap-learn")

## 4. Density Distributions per Group and K

In [ ]:
# Load one CSV per dataset (all K values and PCA/raw in the same file)
density_frames = []
for ds in DATASETS:
    path = os.path.join(EMBEDDINGS_ROOT, ds, MODEL_SLUG, "densities.csv")
    if os.path.exists(path):
        df = pd.read_csv(path)
        df["dataset"] = ds
        density_frames.append(df)

df_combined = pd.concat(density_frames, ignore_index=True)
print(f"Total rows: {len(df_combined):,}")
print("Columns:", df_combined.columns.tolist())

In [ ]:
# Distribution of density_all for largest K — raw vs PCA side by side
k_plot = max(K_VALUES)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for ax, space in zip(axes, ["raw", "pca"]):
    col = f"density_k{k_plot}_all" if space == "raw" else f"density_pca_k{k_plot}_all"
    if col not in df_combined.columns:
        ax.set_title(f"{space} — column not found")
        continue
    for grp, sub in df_combined.groupby("group"):
        ax.hist(sub[col], bins=60, alpha=0.5, label=grp, density=True)
    ax.set_xlabel(f"Log-density vs all  ({space})")
    ax.set_ylabel("Density")
    ax.set_title(f"K={k_plot} — {space} embeddings")
    ax.legend(fontsize=7, ncol=3)

plt.tight_layout()
plt.show()

## 5. Density Ratio (Russian vs All)

In [ ]:
# Density ratio per K and embedding space (raw vs PCA)
n_rows, n_cols = len(K_VALUES), 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5 * n_rows), sharey=False)
if n_rows == 1:
    axes = [axes]

for row, k in enumerate(K_VALUES):
    for col, space in enumerate(["raw", "pca"]):
        ax = axes[row][col]
        ratio_col = f"density_k{k}_ratio" if space == "raw" else f"density_pca_k{k}_ratio"
        if ratio_col not in df_combined.columns:
            ax.set_title(f"K={k} {space} — no ratio column")
            continue
        for grp, sub in df_combined.groupby("group"):
            log_ratio = np.log(sub[ratio_col].replace(0, np.nan).dropna())
            ax.hist(log_ratio, bins=50, alpha=0.5, label=grp, density=True)
        ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
        ax.set_xlabel("Log density ratio (Russian / All)")
        ax.set_title(f"K={k} — {space} embeddings")

axes[0][0].set_ylabel("Density")
axes[-1][-1].legend(fontsize=7, ncol=2)
plt.suptitle("Density ratio: Russian vs All  (raw and PCA)", y=1.01)
plt.tight_layout()
plt.show()

## 6. Cross-Group Density Heatmap

Mean log-density of each group's samples against every other group's reference set.

In [ ]:
# Cross-group density heatmap — raw and PCA side by side for largest K
k_heat = max(K_VALUES)
groups = sorted(df_combined["group"].unique())

fig, axes = plt.subplots(1, 2, figsize=(22, 9))
for ax, space in zip(axes, ["raw", "pca"]):
    prefix = f"density_k{k_heat}_" if space == "raw" else f"density_pca_k{k_heat}_"
    group_cols = [c for c in df_combined.columns if c.startswith(prefix) and not c.endswith(("_all", "_ratio"))]
    col_labels = [c.replace(prefix, "") for c in group_cols]

    matrix = pd.DataFrame(index=groups, columns=col_labels, dtype=float)
    for grp in groups:
        sub = df_combined[df_combined["group"] == grp]
        for col, lbl in zip(group_cols, col_labels):
            if col in sub.columns:
                matrix.loc[grp, lbl] = sub[col].mean()

    sns.heatmap(
        matrix.astype(float), ax=ax,
        annot=True, fmt=".1f", cmap="viridis",
        linewidths=0.5, annot_kws={"size": 7},
        cbar_kws={"label": "Mean log-density"},
    )
    ax.set_xlabel("Target group", fontweight="bold")
    ax.set_ylabel("Source group", fontweight="bold")
    ax.set_title(f"K={k_heat} — {space} embeddings")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")

plt.suptitle("Cross-group density heatmap", y=1.01)
plt.tight_layout()
plt.show()